<a href="https://colab.research.google.com/github/kjahan/armory/blob/main/notebooks/t5_paraphraser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Paraphrase-Generation

Model: `Vamsi/T5_Paraphrase_Paws`

Ref: https://huggingface.co/Vamsi/T5_Paraphrase_Paws


## Install transformers with sentencepiece

We ran into the following issue so we need to install `sentencepiece` to resolve it:

ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece installed to convert a slow tokenizer to a fast one.



In [1]:
!pip install transformers[sentencepiece]

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 4.4 MB 5.3 MB/s 
     |████████████████████████████████| 596 kB 52.5 MB/s 
     |████████████████████████████████| 6.6 MB 44.7 MB/s 
     |████████████████████████████████| 86 kB 6.4 MB/s 
     |████████████████████████████████| 1.2 MB 52.4 MB/s 
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 3.13
    Uninstalling PyYAML-3.13:
      Successfully uninstalled PyYAML-3.13


## Paraphrase any question with T5 (Text-To-Text Transfer Transformer)

In [2]:
import torch
from transformers import T5ForConditionalGeneration,T5Tokenizer


def set_seed(seed):
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

set_seed(42)

model_2 = T5ForConditionalGeneration.from_pretrained('ramsrigouthamg/t5_paraphraser')
tokenizer_2 = T5Tokenizer.from_pretrained('ramsrigouthamg/t5_paraphraser')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print ("device ",device)
model_2 = model_2.to(device)

Downloading:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/850M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/773k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.74k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

device  cuda


In [3]:
def generate_phraphrases(sentence, params):
  text =  "paraphrase: " + sentence # + " </s>"
  encoding = tokenizer_2.encode_plus(text, padding=True, return_tensors="pt")
  input_ids, attention_masks = encoding["input_ids"].to(device), encoding["attention_mask"].to(device)


  # set top_k = 50 and set top_p = 0.95 and num_return_sequences = 3
  beam_outputs = model_2.generate(
      input_ids=input_ids, 
      attention_mask=attention_masks,
      do_sample=params["do_sample"],
      max_length=params["max_length"],
      top_k=params["top_k"],
      top_p=params["top_p"],
      early_stopping=params["early_stopping"],
      num_return_sequences=params["num_return_sequences"]
  )


  print ("\nOriginal Question ::")
  print (sentence)
  print ("\n")
  print ("Paraphrased Questions :: ")
  final_outputs =[]
  for beam_output in beam_outputs:
      sent = tokenizer_2.decode(beam_output, skip_special_tokens=True, clean_up_tokenization_spaces=True)
      if sent.lower() != sentence.lower() and sent not in final_outputs:
          final_outputs.append(sent)

  for i, final_output in enumerate(final_outputs):
      print("{}: {}".format(i, final_output))

In [4]:
params = {}
params["do_sample"] = True
params["max_length"] = 256
params["top_k"] = 50
params["top_p"] = 0.98
params["early_stopping"] = True
params["num_return_sequences"] = 10

sentence = "Gold is just bitcoin that can't be sent over the internet."
# sentence = "Which investment is better during stagflation: stocks, gold, commidity, bitcoin or real estate?"
# sentence = "Which course should I take to get started in data science?"
# sentence = "What are the ingredients required to bake a perfect cake?"
# sentence = "What is the best possible approach to learn aeronautical engineering?"
# sentence = "Do apples taste better than oranges in general?"

generate_phraphrases(sentence, params)


Original Question ::
Gold is just bitcoin that can't be sent over the internet.


Paraphrased Questions :: 
0: How is pure gold used? Gold could be sent over the internet.
1: Gold isn't a currency and can only be sent over a wire.
2: I know Gold is a symbol of what Bitcoin doesn't work in.
3: What is a bitcoin or gold that can't be sent or downloaded over the Internet?
4: Is gold a legal issue and can only be sent over the internet?
5: Gold is just bitcoins that can't be sent over the internet.
6: Gold is just bitcoin, and can't be sent over the internet. Is it safe to send bitcoins to other countries?
7: Gold is just the bitcoin that can't be sent over the internet.
8: Gold is a digital currency, just like bitcoins, which cannot be sent by any means through the internet.
9: Gold - Bitcoin is a fiat currency in a world where nothing is going to be sent over the internet.


In [5]:
params = {}
params["do_sample"] = True
params["max_length"] = 256
params["top_k"] = 50
params["top_p"] = 0.98
params["early_stopping"] = True
params["num_return_sequences"] = 10

# Ref: https://cs.uwaterloo.ca/~sgorbuno/courses/858f20/index.html
sentence = "Blockchain technologies are quickly gaining popularity and enable the borderless and frictionless transfer of assets. In this course, we will study blockchain fundamentals and applications built on top of them. Topics include signature schemes, commitment schemes, multi-party computation, zero-knowledge proofs, consensus mechanisms, smart contracts, incentive mechanisms, applications of blockchains such as cryptocurrencies, and the legal framework around them."

generate_phraphrases(sentence, params)


Original Question ::
Blockchain technologies are quickly gaining popularity and enable the borderless and frictionless transfer of assets. In this course, we will study blockchain fundamentals and applications built on top of them. Topics include signature schemes, commitment schemes, multi-party computation, zero-knowledge proofs, consensus mechanisms, smart contracts, incentive mechanisms, applications of blockchains such as cryptocurrencies, and the legal framework around them.


Paraphrased Questions :: 
0: In this course, we will study blockchain fundamentals and applications built on top of them. Topics include signature schemes, commitment schemes, multi-party computation, zero-knowledge proofs, consensus mechanisms, smart contracts, incentives, applications of blockchains such as cryptocurrencies, and the legal framework around them.
1: Blockchain technologies are rapidly gaining popularity and enable the borderless and frictionless transfer of assets. In this course we will s

In [ ]:
# Ref: https://www.reuters.com/article/usa-bitcoin-environment/update-1-insight-coal-to-crypto-the-gold-rush-bringing-bitcoin-miners-to-kentucky-idUKL5N2VO4WT
sentence = "As part of Kentucky’s drive to woo bitcoin miners, legislation written by Smith allows miners who invest more than a million dollars in the state to have their sales taxes waived."

params = {}

params["do_sample"] = True
params["max_length"] = 128
params["top_k"] = 50
params["top_p"] = 0.95
params["early_stopping"] = True
params["num_return_sequences"] = 5

generate_phraphrases(sentence, params)


Original Question ::
As part of Kentucky’s drive to woo bitcoin miners, legislation written by Smith allows miners who invest more than a million dollars in the state to have their sales taxes waived.


Paraphrased Questions :: 
0: As part of Kentucky’s drive to woo bitcoin miners, legislation written by Smith allows miners who invest over a million dollars in the state to have their sales taxes waived.
1: Will Kentucky's election to become the next gold producer have a tax-free transaction?
2: As part of Kentucky’s drive to woo bitcoin miners, legislation written by Smith allows miners to invest more than 1 million dollars in the state to have their sales taxes waived.
3: Currently, Kentucky has no sales tax for its bitcoin operations. But now it can get rid of it. For a small business, it would make sense to bring bitcoin assets to Kentucky.
4: As part of Kentucky’s drive to woo Bitcoin miners, legislation written by Smith allows miners who invest more than one million dollars in th